# Import Packages

In [ ]:
import sys
import sklearn
import numpy as np
import scipy
import joblib
from pathlib import Path
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import precision_recall_fscore_support

from transformers import set_seed
set_seed(42)

In [2]:
print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Logistic regression training device: CPU")

Python: 3.14.7
scikit-learn: 1.9.0
NumPy: 2.5.2
SciPy: 1.18.1
Logistic regression training device: CPU


# Load dataset, and filter/split it

In [3]:
dataset = load_dataset("coastalcph/tydi_xor_rc")


df_train = dataset["train"].to_pandas()
df_validation = dataset["validation"].to_pandas()

# Selected languages
langlst = ['ko','ar','te']

# Training and Validation data for selected languages
df_train_filtered = df_train[df_train['lang'].isin(langlst)]
df_validation_filtered = df_validation[df_validation['lang'].isin(langlst)]
 
#Training and Validation data for each language
df_train_ar = df_train[(df_train['lang'] == "ar")]
df_train_ko = df_train[(df_train['lang'] == "ko")]
df_train_te = df_train[(df_train['lang'] == "te")]

df_validation_ar = df_validation[(df_validation['lang'] == "ar")]
df_validation_ko = df_validation[(df_validation['lang'] == "ko")]
df_validation_te = df_validation[(df_validation['lang'] == "te")]


tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-multilingual-cased", use_fast=True
)


# Logistic-regression baseline

In [4]:
train_texts = df_train_filtered["question"] + " " + df_train_filtered["context"]
validation_texts = df_validation_filtered["question"] + " " + df_validation_filtered["context"]

vectorizer = CountVectorizer(min_df=2)
X_train = vectorizer.fit_transform(train_texts)
X_validation = vectorizer.transform(validation_texts)
y_train = df_train_filtered["answerable"]
y_validation = df_validation_filtered["answerable"]

logistic_baseline = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
)
logistic_baseline.fit(X_train, y_train)

model_dir = Path("week39-logistic-regression")
model_dir.mkdir(exist_ok=True)
joblib.dump(
    {"model": logistic_baseline, "vectorizer": vectorizer},
    model_dir / "model.joblib",
)

validation_predictions = logistic_baseline.predict(X_validation)



results = []

for language in ["all", "ar", "ko", "te"]:
    if language == "all":
        true_labels = y_validation
        predicted_labels = validation_predictions
    else:
        language_rows = df_validation_filtered["lang"] == language
        true_labels = y_validation[language_rows]
        predicted_labels = validation_predictions[language_rows]

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, predicted_labels, labels=[False, True]
    )
    results.append([
        language,
        len(true_labels),
        precision[0], recall[0], f1[0],
        precision[1], recall[1], f1[1],
        (f1[0] + f1[1]) / 2,
    ])

results = pd.DataFrame(
    results,
    columns=[
        "language", "n",
        "unanswerable_P", "unanswerable_R", "unanswerable_F1",
        "answerable_P", "answerable_R", "answerable_F1", "macro_F1",
    ],
)
results.round(3)

,language,n,unanswerable_P,unanswerable_R,unanswerable_F1,answerable_P,answerable_R,answerable_F1,macro_F1
0,all,1155,0.800,0.341,0.479,0.900,0.986,0.941,0.710
1,ar,415,0.900,0.865,0.882,0.981,0.986,0.984,0.933
2,ko,356,0.000,0.000,0.000,0.946,0.997,0.971,0.486
3,te,384,0.579,0.118,0.196,0.775,0.973,0.863,0.530


# Recurrent classifier


In [5]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

train_tokens = tokenizer(
    df_train_filtered["question"].tolist(),
    df_train_filtered["context"].tolist(),
    padding="max_length", truncation="only_second", return_tensors="pt",
)
validation_tokens = tokenizer(
    df_validation_filtered["question"].tolist(),
    df_validation_filtered["context"].tolist(),
    padding="max_length", truncation="only_second", return_tensors="pt",
)

train_labels = torch.tensor(df_train_filtered["answerable"].astype(int).to_numpy())

train_data = TensorDataset(
    train_tokens["input_ids"], train_tokens["attention_mask"], train_labels
)
validation_data = TensorDataset(
    validation_tokens["input_ids"], validation_tokens["attention_mask"]
)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
validation_loader = DataLoader(validation_data, batch_size=32)

## RNN class definition and evaluation function

Implementation of the RNN model, and the method for evaluating the output of the model. 

In [6]:
class RecurrentAnswerabilityClassifier(nn.Module):
	def __init__(self, vocabulary_size, padding_index, embedding_size=64, hidden_size=64):
		super().__init__()
		self.embedding = nn.Embedding(vocabulary_size, embedding_size, padding_idx=padding_index)
		self.lstm = nn.LSTM(embedding_size, hidden_size, batch_first=True)
		self.classifier = nn.Linear(hidden_size, 2)

	def forward(self, token_ids, attention_mask):
		embeddings = self.embedding(token_ids)
		outputs, _ = self.lstm(embeddings)
		
		last_tokens = (attention_mask.sum(dim=1) - 1).tolist()
		last_outputs = []
		
		for example_number in range(len(token_ids)):
			last_outputs.append(outputs[example_number, last_tokens[example_number]])
		return self.classifier(torch.stack(last_outputs))

	def train_model(self, train_loader, loss_function, optimizer, device, epochs=5):
		for epoch in range(epochs):
			self.train()
			total_loss = 0
			total_examples = 0
		
			for token_ids, attention_mask, labels in train_loader:
				token_ids = token_ids.to(device)
				attention_mask = attention_mask.to(device)
				labels = labels.to(device)
		
				optimizer.zero_grad()
			
				logits = self(token_ids, attention_mask)
				loss = loss_function(logits, labels)
		
				loss.backward()
				optimizer.step()
		
				total_loss += loss.item() * len(labels)
				total_examples += len(labels)
		
			print(f"Epoch {epoch + 1}: loss = {total_loss / total_examples:.4f}")  

	
	def predict(self, data_loader, device):
		self.eval()
		predictions = []

		with torch.no_grad():
			for token_ids, attention_mask in data_loader:
				logits = self(token_ids.to(device), attention_mask.to(device))
				predictions.extend(logits.argmax(dim=1).bool().cpu().tolist())

		return predictions



def evaluate_model(predictions, validation_data, languages=["all", "ar", "ko", "te"]):
	results = validation_data.copy()
	results["prediction"] = predictions

	evaluation_results = []

	for language in languages:
		group = results if language == "all" else results[results["lang"] == language]

		true_labels = group["answerable"]
		predicted_labels = group["prediction"]

		precision, recall, f1, _ = precision_recall_fscore_support(
			true_labels, predicted_labels, labels=[False, True]
		)

		evaluation_results.append([
			language,
			len(true_labels),
			precision[0], recall[0], f1[0],
			precision[1], recall[1], f1[1],
			(f1[0] + f1[1]) / 2,
		])

	return pd.DataFrame(
		evaluation_results,
		columns=[
			"language", "n",
			"unanswerable_P", "unanswerable_R", "unanswerable_F1",
			"answerable_P", "answerable_R", "answerable_F1", "macro_F1",
		],
	)


## Training of the model

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

recurrent_model = RecurrentAnswerabilityClassifier(
    len(tokenizer), tokenizer.pad_token_id
).to(device)

unanswerable_count = (train_labels == 0).sum().item()
answerable_count = (train_labels == 1).sum().item()

class_weights = torch.tensor([
    len(train_labels) / (2 * unanswerable_count),
    len(train_labels) / (2 * answerable_count),
], device=device)

loss_function = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(recurrent_model.parameters(), lr=0.001)

recurrent_model.train_model(
    train_loader,
    loss_function,
    optimizer,
    device,
    epochs=5
)

Epoch 1: loss = 0.6818
Epoch 2: loss = 0.6240
Epoch 3: loss = 0.5023
Epoch 4: loss = 0.3412
Epoch 5: loss = 0.2049


## Predictions and results

The predictions of the model, for answerable/unanswerable precision, recall and F1 scores. Together with a combined Macro-F1 score.

In [8]:
recurrent_predictions = recurrent_model.predict(validation_loader, device)

recurrent_results = evaluate_model(
    recurrent_predictions,
    df_validation_filtered
)

recurrent_results.round(3)


,language,n,unanswerable_P,unanswerable_R,unanswerable_F1,answerable_P,answerable_R,answerable_F1,macro_F1
0,all,1155,0.347,0.427,0.383,0.901,0.867,0.884,0.633
1,ar,415,0.234,0.288,0.259,0.895,0.865,0.880,0.569
2,ko,356,0.151,0.421,0.222,0.964,0.866,0.912,0.567
3,te,384,0.553,0.505,0.528,0.846,0.869,0.858,0.693


# Validation Errors

The number of wrong predictions by the models, categorized into false positives, and false negatives, split by languages. 

Additionally, 12 specific examples of wrong predictions. 6 for each model, 

In [9]:
errors_by_model = []

for model, predictions in [
    ("logistic regression", validation_predictions),
    ("recurrent classifier", recurrent_predictions),
]:
    errors = df_validation_filtered[["lang", "question", "context", "answerable", "answer", "answer_start"]].copy()
    errors["prediction"] = predictions
    errors = errors[errors["prediction"] != errors["answerable"]].copy()
    errors["model"] = model
    errors["error_type"] = "false positive"
    errors.loc[errors["answerable"], "error_type"] = "false negative"
    errors_by_model.append(errors)

all_errors = pd.concat(errors_by_model)

error_counts = all_errors.groupby(["model", "lang", "error_type"]).size()
print(error_counts)



selected_errors = []

for model in ["logistic regression", "recurrent classifier"]:
    for language in ["ar", "ko", "te"]:
        for error_type in ["false positive", "false negative"]:
            matching_errors = all_errors[
                (all_errors["model"] == model)
                & (all_errors["lang"] == language)
                & (all_errors["error_type"] == error_type)
            ]
            if len(matching_errors) > 0:
                selected_errors.append(matching_errors.iloc[[0]])

selected_errors = pd.concat(selected_errors).reset_index()
selected_errors = selected_errors.rename(columns={"index": "validation_index"})
print(f"Selected {len(selected_errors)} errors")
pd.set_option("display.max_colwidth", 300)
selected_errors[[
    "validation_index", "model", "lang", "error_type",
    "question", "context", "answerable", "prediction",
    "answer", "answer_start",
]]

model                 lang  error_type    
logistic regression   ar    false negative     5
                            false positive     7
                      ko    false negative     1
                            false positive    19
                      te    false negative     8
                            false positive    82
recurrent classifier  ar    false negative    49
                            false positive    37
                      ko    false negative    45
                            false positive    11
                      te    false negative    38
                            false positive    46
dtype: int64
Selected 12 errors


,validation_index,model,lang,error_type,question,context,answerable,prediction,answer,answer_start
0,1714,logistic regression,ar,false positive,هل يمكن حماية الأطفال من التعرض إلي التحرش الجنسي ؟,"In the United States growing awareness of child sexual abuse has sparked an increasing number of civil lawsuits for monetary damages stemming from such incidents. Increased awareness of child sexual abuse has encouraged more victims to come forward, whereas in the past victims often kept their a...",False,True,no,-1
1,1478,logistic regression,ar,false negative,هل مصرهي دولة نامية؟,"In recent years, new terms have emerged to describe the largest developing countries such as BRIC that stands for Brazil, Russia, India, and China, along with ""BRICET"" (BRIC + Eastern Europe and Turkey), ""BRICS"" (BRIC + South Africa), ""BRICM"" (BRIC + Mexico), MINT (Mexico, Indonesia, Nigeria and...",True,False,Egypt,475
2,364,logistic regression,ko,false positive,시차는 중력과 관련이 있는가?,"Gravitational time dilation was first described by Albert Einstein in 1907 as a consequence of special relativity in accelerated frames of reference. In general relativity, it is considered to be a difference in the passage of proper time at different positions as described by a metric tensor of...",False,True,no,-1
3,1214,logistic regression,ko,false negative,디젤 자동차는 누가 개발했는가?,"The diesel car markets are the same ones who pioneered various developments (Mercedes-Benz, BMW, Peugeot/Citroën, Fiat, Alfa Romeo, Volkswagen Group) There were also small diesel engines produced in England by British Leyland and Perkins. For reasons of economy the petrol BMC ""B"" series engine w...",True,False,Perkins Engines,720
4,2911,logistic regression,te,false positive,మలేరియా వ్యాధి కి మందు కనిపెట్టిన శాస్త్రవేత్త ఎవరు?,"Malaria is a disease spread by mosquitoes. Malaria is transmitted when the parasite enters the blood of a person. Parasites depend on the humans they live with for their food. ""Charles Louis Alphonse Lavren"", a French defense physician, was awarded the Nobel Prize in 1907 for discovering how mal...",False,True,Hans Andersag,-1
5,93,logistic regression,te,false negative,తెలుగు పంచాంగం ప్రకారం నూతన సంవత్సరం ఏ ఇంగ్లీష్ నెలలో ప్రారంభమవుతుంది?,The Telugu year is the calendar year for the Telugu speaking people of . Each Yuga (Era) has a cycle of 60 years. Each year of Ugadi year has a specific name in Panchangam (Astrnomical calendar) based on astrological influences and the name of the year would represent the character of that year....,True,False,March or April,560
6,1417,recurrent classifier,ar,false positive,هل زار ابن بطوطة اليمن؟,"After the ""hajj"" in either 1328 or 1330, he made his way to the port of Jeddah on the Red Sea coast. From there he followed the coast in a series of boats making slow progress against the prevailing south-easterly winds. Once in Yemen he visited Zabīd and later the highland town of Ta'izz, where...",False,True,no,-1
7,1413,recurrent classifier,ar,false negative,اين عاش نيوتن؟,"From age 12 to age 17, Newton resided with William Clarke, apothecary, in Grantham, where he acquired his interest in chemistry. While living with the Clarke family, Newton was educated at The King's School, Grantham (where his signature can still be seen upon a library window sill). He spent mu...",True,False,Grantham,74
8,368,recurrent classifier,ko,false positive,맹장 없이 살 수 있을까?,"In modern humans, the vermiform appendix is a vestige of a redundant organ that in ancestral species had digestive functions, much as it still does in extant species in which intestinal flora hydrolyze cellulose and similar indigestible plant materials. Some herbivorous animals, such as rabbits,...",False,True,no,-1
9,361,recurrent classifier,ko,false negative,지구에서 가장 가까운 행성은 무엇인가?,"Mercury attains inferior conjunction (nearest approach to Earth) every 116 Earth days on average, but this interval can range from 105 days to 129 days due to the planet's eccentric orbit. Mercury can come as near 